In [ ]:
import torch
import shutil
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    CLIPVisionModel,
    CLIPVisionConfig,
    CLIPImageProcessor,
    LlavaConfig,
    LlavaProcessor,
    LlavaForConditionalGeneration,
)

In [ ]:
MODELS_ROOT = Path('/mnt/models')
# MODELS_ROOT = Path('/nas_train/app.e0016372/models')

VISION_MODEL_PATH = MODELS_ROOT / 'clip-vit-large-patch14-336'
TEXT_MODEL_PATH = MODELS_ROOT / 'Qwen3-4B-Instruct-2507'
TOKENIZER_PATH = MODELS_ROOT / 'Qwen3-VL-4B-Instruct'
OUTPUT_DIR = MODELS_ROOT / 'Llava-Qwen3-4B-Instruct'

IMAGE_TOKEN_ID = 151655
IMAGE_TOKEN_STR = '<|image_pad|>'

In [ ]:
def clear_output_dir():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

def build_llava_model():
    vision_model = CLIPVisionModel.from_pretrained(VISION_MODEL_PATH, torch_dtype=torch.bfloat16)
    text_model = AutoModelForCausalLM.from_pretrained(TEXT_MODEL_PATH, torch_dtype=torch.bfloat16)

    config = LlavaConfig(
        vision_config=vision_model.config,
        text_config=text_model.config,
        image_token_index=IMAGE_TOKEN_ID,
    )
    llava_model = LlavaForConditionalGeneration(config)
    llava_model.vision_tower = vision_model
    llava_model.language_model = text_model
    llava_model.save_pretrained(OUTPUT_DIR)

def build_llava_processor():
    vision_config = CLIPVisionConfig.from_pretrained(VISION_MODEL_PATH)
    image_processor = CLIPImageProcessor.from_pretrained(VISION_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

    processor = LlavaProcessor(
        image_processor=image_processor,
        tokenizer=tokenizer,
        patch_size=vision_config.patch_size,
        vision_feature_select_strategy='default',
        chat_template=tokenizer.chat_template,
        image_token=IMAGE_TOKEN_STR,
    )
    processor.save_pretrained(OUTPUT_DIR)

In [ ]:
clear_output_dir()
build_llava_model()
build_llava_processor()